# PricePilot AI - Model Evaluation and Analysis

## 1. Project Objective
PricePilot AI helps retail managers figure out the best price for their products. 
It looks at product details (like brand and category) and what is happening in the market right now (like demand, season, and competitor prices).
The goal is to suggest a price that makes good money but is still fair for customers.

## 2. Dataset Overview
We are using a large spreadsheet of past sales data to teach the AI.
- **Number of records**: Over 170,000 past sales.
- **Information included**: Product details, stock levels, and competitor prices.
- **What we want to predict**: `current_price` (the final selling price).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('default')
sns.set_theme(style='whitegrid')

dataset_path = 'data/retail_price_optimization_dataset.csv'
df = pd.read_csv(dataset_path)

print(f"Dataset loaded successfully: {len(df)} records and {len(df.columns)} columns.")
display(df.head())

## 3. Business Features
To keep the AI simple and easy to understand, we only give it 13 basic business facts to look at:

**Product Facts**
- Name, Brand, and Category.
- Is the product new or old? (`product_lifecycle`)
- How good is it? (`average_rating`, `historical_sales`)
- What does it cost us? (`base_price`, `cost_price`)

**Today's Market**
- What are competitors charging? (`competitor_price`)
- Do people really want it right now? (`demand_index`)
- Do we have a lot in the warehouse? (`inventory_level`)
- Is there a sale or special season? (`promotion_type`, `season`)

**Target**
- `current_price`

## 4. Preprocessing
Before the AI can learn, we have to clean up the data:
1. We keep only the 13 facts listed above.
2. We fill in any blank spaces.
3. **Change words to numbers**: AI only reads numbers, so we change words (like 'Winter') into a number code.
4. **Split the data**: We hide 20% of the data from the AI so we can test it like a final exam later.

In [ ]:
preprocessor = joblib.load('models/preprocessor.joblib')
model = joblib.load('models/price_model.joblib')

allowed_features = [
    'product_name', 'brand', 'category', 'base_price', 'cost_price',
    'competitor_price', 'demand_index', 'inventory_level', 'promotion_type',
    'season', 'historical_sales', 'average_rating', 'product_lifecycle'
]
target = 'current_price'

X = df[allowed_features]
y = df[target]

X_processed = preprocessor.transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
print("Preprocessing Complete.")
print(f"Training set: {X_train.shape[0]} records. Testing set (Final Exam): {X_test.shape[0]} records.")

## 5. Model Training & 6. Evaluation Metrics
We picked a very simple AI model called **Linear Regression**. 
We chose this because it is easy to explain. It tells us exactly how many rupees to add or subtract for things like 'High Demand' or 'Winter Season'.

**How we grade the AI's final exam**:
- **MAE**: On average, how many rupees is the AI's guess wrong by?
- **RMSE**: Just like MAE, but it punishes the AI more for really huge mistakes.
- **R² Score**: A score out of 1.0 that tells us how perfect the guesses are (closer to 1.0 is better).

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- Evaluation Metrics ---")
print(f"MAE:  ₹ {mae:,.2f}")
print(f"RMSE: ₹ {rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

## 7. Visualizations
These charts show how the business facts connect to the price, and how well our AI performs.

In [ ]:
# 7.1 Actual vs Predicted Scatter Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='indigo')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price (₹)')
plt.ylabel('AI Predicted Price (₹)')
plt.title('Actual vs. Predicted Price')
plt.show()

In [ ]:
# 7.2 Residual Plot
residuals = y_test - y_pred
plt.figure(figsize=(8, 6))
sns.histplot(residuals, bins=50, kde=True, color='purple')
plt.title('How often is the AI wrong? (Error Distribution)')
plt.xlabel('Prediction Error (₹)')
plt.show()

In [ ]:
# 7.3 Correlation Heatmap (Numeric Features)
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('How are facts related to each other?')
plt.show()

In [ ]:
# 7.4 Price Distribution by Category
plt.figure(figsize=(10, 6))
sns.boxplot(x='category', y='current_price', data=df, palette='Set2')
plt.title('Price Spread by Category')
plt.xticks(rotation=45)
plt.ylabel('Current Price (₹)')
plt.xlabel('Category')
plt.show()

In [ ]:
# 7.5 Demand vs Price
plt.figure(figsize=(8, 6))
sns.scatterplot(x='demand_index', y='current_price', data=df, alpha=0.5, color='teal')
plt.title('High Demand = Higher Price?')
plt.xlabel('Demand Index')
plt.ylabel('Current Price (₹)')
plt.show()

In [ ]:
# 7.6 Competitor Price vs Current Price
plt.figure(figsize=(8, 6))
sns.scatterplot(x='competitor_price', y='current_price', data=df, alpha=0.5, color='coral')
plt.title('Do we follow competitor prices?')
plt.xlabel('Competitor Price (₹)')
plt.ylabel('Current Price (₹)')
plt.show()

In [ ]:
# 7.7 Average Price by Season
season_avg = df.groupby('season')['current_price'].mean().reset_index()
plt.figure(figsize=(8, 6))
sns.barplot(x='season', y='current_price', data=season_avg, palette='viridis')
plt.title('Does Season change the average price?')
plt.ylabel('Average Price (₹)')
plt.xlabel('Season')
plt.show()

In [ ]:
# 7.8 How Each Business Factor Influences Price
feature_names = preprocessor.get_feature_names_out()
# Clean up names for presentation
clean_names = []
for f in feature_names:
    f = f.replace('cat__', '').replace('num__', '')
    f = f.replace('promotion_type_', 'Promotion: ')
    f = f.replace('season_', 'Season: ')
    clean_names.append(f)

coefficients = pd.DataFrame({'Feature': clean_names, 'Impact (₹)': model.coef_})
top_positive = coefficients.sort_values(by='Impact (₹)', ascending=False).head(10)
top_negative = coefficients.sort_values(by='Impact (₹)').head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x='Impact (₹)', y='Feature', data=top_positive, ax=axes[0], palette='Greens_r')
axes[0].set_title('Top Price Enhancers (+)')
sns.barplot(x='Impact (₹)', y='Feature', data=top_negative, ax=axes[1], palette='Reds_r')
axes[1].set_title('Top Price Reducers (-)')
plt.tight_layout()
plt.show()

## 8. Example Prediction
Let's test the AI on one specific product. We give it the product details, and it tells us what the final price should be.

In [ ]:
sample = pd.DataFrame([{
    'product_name': 'Coach Tabby Shoulder Bag',
    'brand': 'Coach',
    'category': 'Accessories',
    'base_price': 23580.0,
    'cost_price': 15000.0,
    'competitor_price': 17726.0,
    'demand_index': 120.0,
    'inventory_level': 80,
    'promotion_type': 'No Promotion',
    'season': 'Winter',
    'historical_sales': 1200,
    'average_rating': 4.5,
    'product_lifecycle': 'Maturity'
}])

sample_processed = preprocessor.transform(sample)
predicted = model.predict(sample_processed)[0]

print("=== Example Business Scenario ===")
print(f"Product Name:      {sample['product_name'][0]}")
print(f"Base Price:        ₹ {sample['base_price'][0]:,.0f}")
print(f"Competitor Price:  ₹ {sample['competitor_price'][0]:,.0f}")
print(f"Demand Index:      {sample['demand_index'][0]} (High)")
print(f"Inventory Level:   {sample['inventory_level'][0]} (Moderate)")
print(f"Season:            {sample['season'][0]}")
print("---------------------------------")
print(f"AI Predicted Price: ₹ {predicted:,.0f}")
print("---------------------------------")
print("Business Explanation:")
print("✓ Demand is High")
print("✓ Inventory is Moderate")
print("✓ Competitor price is higher")
print("✓ Historical sales are strong")
print("\nRecommendation: Increase Price")

## 9. Conclusion
PricePilot AI thinks just like an expert pricing manager. By looking at basic product facts and what the market is doing right now, it can instantly guess the smartest selling price. This helps the business stay competitive and make more money without doing any manual math.